# 📘 Bloque 1: Fundamentos de Machine Learning

**Objetivo:** Entender los tipos de aprendizaje, los algoritmos clásicos y cómo evaluar modelos correctamente.

---

## 1. ¿Qué es Machine Learning?

Machine Learning (ML) es la rama de la IA que permite a los sistemas **aprender patrones a partir de datos** sin ser programados explícitamente para cada caso.

### Tipos de aprendizaje:

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Supervisado** | Aprende con datos etiquetados (X → y) | Clasificar emails como spam o no spam |
| **No supervisado** | Encuentra patrones sin etiquetas | Agrupar clientes por comportamiento |
| **Por refuerzo** | Un agente aprende por recompensas/penalizaciones | AlphaGo, coches autónomos |

---

## 2. Pipeline básico de ML con scikit-learn

El flujo estándar de cualquier proyecto de ML es siempre el mismo:

In [ ]:
# Instalación (ejecuta solo si no tienes las librerías)
# !pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. Cargar datos
iris = load_iris()
X, y = iris.data, iris.target

print(f"Shape de X: {X.shape}")  # (150, 4) -> 150 muestras, 4 features
print(f"Clases: {iris.target_names}")
print(f"Primeras filas:\n{pd.DataFrame(X, columns=iris.feature_names).head()}")

In [ ]:
# 2. Split train/test (regla general: 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# 3. Escalado (muy importante para muchos algoritmos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit+transform en train
X_test_scaled = scaler.transform(X_test)        # solo transform en test (¡nunca fit!)

## 3. Algoritmos clásicos

### 3.1 Regresión Logística (clasificación)
> A pesar del nombre, es un clasificador. Aprende una frontera lineal entre clases.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))

### 3.2 Random Forest (árbol de decisión con ensemble)
> Crea muchos árboles y combina sus predicciones. Robusto y fácil de usar.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=iris.target_names))

# Importancia de features
feat_importance = pd.Series(rf_model.feature_importances_, index=iris.feature_names)
feat_importance.sort_values().plot(kind='barh', title='Importancia de Features')
plt.tight_layout()
plt.show()

### 3.3 SVM — Support Vector Machine
> Encuentra el hiperplano que maximiza el margen entre clases. Muy efectivo en espacios de alta dimensión.

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("=== SVM ===")
print(classification_report(y_test, y_pred_svm, target_names=iris.target_names))

## 4. Evaluación de modelos

### Métricas principales para clasificación:

| Métrica | Fórmula | Cuándo usarla |
|---|---|---|
| **Accuracy** | TP+TN / total | Clases balanceadas |
| **Precision** | TP / (TP+FP) | Minimizar falsos positivos |
| **Recall** | TP / (TP+FN) | Minimizar falsos negativos (ej: enfermedades) |
| **F1-Score** | 2·(P·R)/(P+R) | Balance entre precision y recall |
| **ROC-AUC** | Área bajo curva ROC | Comparar modelos con clases desbalanceadas |

In [ ]:
# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Matriz de Confusión — Random Forest')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## 5. Overfitting vs Underfitting

> Este es uno de los conceptos más importantes en ML.

- **Underfitting**: El modelo es demasiado simple, no aprende ni en train ni en test.
- **Overfitting**: El modelo memoriza train pero falla en test (no generaliza).
- **Buen modelo**: Buena métrica tanto en train como en test.

**Solución al overfitting**: más datos, regularización (L1/L2), dropout, cross-validation.

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validation de 5 folds: entrena 5 veces con distintos splits
scores = cross_val_score(rf_model, X, y, cv=5, scoring='f1_macro')

print(f"F1 por fold: {scores.round(3)}")
print(f"F1 medio: {scores.mean():.3f} ± {scores.std():.3f}")

## 6. Comparativa de modelos

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

modelos = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf'),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred, average='macro')
    resultados[nombre] = f1

pd.Series(resultados).sort_values().plot(
    kind='barh', title='F1-Score por Modelo', xlim=(0.8, 1.0), color='steelblue'
)
plt.tight_layout()
plt.show()

---

## ✅ Resumen del bloque

- Aprendiste los **3 tipos de ML**: supervisado, no supervisado, refuerzo
- Implementaste un **pipeline completo**: carga → split → escalar → entrenar → evaluar
- Conoces los **algoritmos clásicos**: Regresión Logística, Random Forest, SVM, KNN
- Entiendes las **métricas**: Accuracy, Precision, Recall, F1, matriz de confusión
- Sabes qué es **overfitting** y cómo detectarlo con cross-validation

---

## ➡️ Siguiente paso

Continúa con el **Bloque 2: Deep Learning con PyTorch** → `02_deep_learning_pytorch.ipynb`